# Data Preprocessing – BLE DoS Detection

Loads the chronologically split train/val CSVs, encodes labels, applies TF‑IDF on `info`  
(fitted on training only), scales numerical features, and saves the processed arrays  
and artifacts for training.

In [1]:
import pandas as pd
import numpy as np
import pickle, os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, save_npz, csr_matrix

## 1. Load train and validation CSV files

In [2]:
train_df = pd.read_csv('../data/raw/ble_train.csv')
val_df   = pd.read_csv('../data/raw/ble_val.csv')
print(f"Train shape: {train_df.shape}, Val shape: {val_df.shape}")
print("Train labels:\n", train_df['type'].value_counts())
print("Val labels:\n", val_df['type'].value_counts())

Train shape: (972706, 5), Val shape: (243178, 5)
Train labels:
 type
DoS       798712
normal    173994
Name: count, dtype: int64
Val labels:
 type
DoS       199679
normal     43499
Name: count, dtype: int64


## 2. Encode labels (fit on train only)

In [3]:
le = LabelEncoder()
le.fit(train_df['type'])

train_df['label'] = le.transform(train_df['type'])
val_df['label']   = le.transform(val_df['type'])

with open('../artifacts/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print("Classes:", le.classes_)

Classes: ['DoS' 'normal']


## 3. TF‑IDF on `info` (fit only on training data)

In [4]:
tfidf = TfidfVectorizer(max_features=50)
X_train_info = tfidf.fit_transform(train_df['info'])
X_val_info   = tfidf.transform(val_df['info'])

with open('../artifacts/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("TF‑IDF shape:", X_train_info.shape)

TF‑IDF shape: (972706, 50)


## 4. Combine with numerical features and scale

In [5]:
# Numerical features
train_num = train_df[['length', 'delta']].values.astype(np.float32)
val_num   = val_df[['length', 'delta']].values.astype(np.float32)

# Scale only numerical part (fit on train)
scaler = StandardScaler()
train_num_scaled = scaler.fit_transform(train_num)
val_num_scaled   = scaler.transform(val_num)

with open('../artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Combine TF‑IDF and scaled numerical
X_train = hstack([X_train_info, csr_matrix(train_num_scaled)]).tocsr()
X_val   = hstack([X_val_info, csr_matrix(val_num_scaled)]).tocsr()

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}")

Train shape: (972706, 52), Val shape: (243178, 52)


## 5. Save processed data

In [6]:
y_train = train_df['label'].values
y_val   = val_df['label'].values

os.makedirs('../data/processed', exist_ok=True)
save_npz('../data/processed/X_train.npz', X_train)
save_npz('../data/processed/X_val.npz', X_val)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy', y_val)

print("✅ Saved train/val arrays in data/processed/")

✅ Saved train/val arrays in data/processed/
